# GreenChanger Machine Learning

In [1]:
import json
import math
import boto3, psycopg
import numpy as np
import pandas as pd
import statsmodels.api as sm
import pickle
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

In [2]:
HOST = "greenshift-dev.cluster-cl6ugmisezq2.ap-southeast-2.rds.amazonaws.com"
USER = "greenshift_api"

token = boto3.client("rds", region_name="ap-southeast-2").generate_db_auth_token(
    DBHostname=HOST, Port=5432, DBUsername=USER, Region="ap-southeast-2")
conn = psycopg.connect(host=HOST, port=5432, dbname="postgres", user=USER,
                       password=token, sslmode="require")

# define query function to database
def query(sql, params=None):
    with conn.cursor() as cur:
        cur.execute(sql, params)
        cols = [d.name for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)

In [3]:
# get the latest dataset version ids for canopy and tree inventory
canopy_version = query("""
    SELECT DISTINCT dataset_version_id FROM latest_city_canopy_snapshots
    WHERE observed_year = 2021""").iloc[0, 0]
tree_version = query("""
    SELECT DISTINCT dataset_version_id FROM latest_city_melbourne_named_tree_inventory
    """).iloc[0, 0]
canopy_version, tree_version

(UUID('bfa4a564-92da-4dd5-8e19-c753bca97eff'),
 UUID('f579bc01-219b-4c6b-9446-4eb8e18bde17'))

In [4]:
SQL = """
WITH hit AS (
  SELECT t.named_tree_id,
         lower(t.scientific_name)     AS species,
         t.year_planted,
         t.precinct,
         p.canopy_snapshot_feature_id AS polygon_id,
         p.calculated_area_m2         AS area_m2
  FROM named_tree_inventory t
  JOIN LATERAL (
    SELECT f.canopy_snapshot_feature_id, f.calculated_area_m2
    FROM canopy_snapshot_feature f
    WHERE f.dataset_version_id = %(canopy_version)s
      AND f.quality_status = 'passed'
      AND ST_Intersects(f.canopy_geometry, t.tree_location)
    LIMIT 1
  ) p ON true
  WHERE t.dataset_version_id = %(tree_version)s
    AND t.quality_status = 'passed'
),
alone AS (
  SELECT polygon_id FROM hit GROUP BY polygon_id HAVING count(*) = 1
)
SELECT h.species, h.year_planted, h.precinct, h.area_m2
FROM hit h JOIN alone USING (polygon_id)
WHERE h.year_planted BETWEEN 2003 AND 2020
"""
df = query(SQL, {"canopy_version": canopy_version, "tree_version": tree_version})
conn.close()
len(df)



7552

In [ ]:
display(df.head(10))

In [ ]:
df["area_m2"] = df["area_m2"].astype(float)
df["age"] = 2021 - df["year_planted"]

df = df[(df["age"] >= 3) & (df["area_m2"] > 0)]
df = df[df["area_m2"] <= 200]
df["log_area"] = np.log(df["area_m2"])

counts = df["species"].value_counts()
df = df[df["species"].isin(counts[counts >= 30].index)]

len(df), df["species"].nunique(), df["age"].min(), df["age"].max()


In [ ]:
# split the dataset into training and testing sets

seed = 42


train, test = train_test_split(df, test_size=0.2, random_state=seed,
                               stratify=df["species"])

len(train), len(test), train["species"].nunique(), test["species"].nunique()

In [ ]:
# quantile regression per species: log(canopy area) = a + b * age
models = {}

# fit p10 / p50 / p90 lines and keep species where all three grow with age
for species, group in train.groupby("species"):
    X = sm.add_constant(group[["age"]])
    fits = {name: sm.QuantReg(group["log_area"], X).fit(q=q)
            for name, q in [("p10", 0.1), ("p50", 0.5), ("p90", 0.9)]}
    if all(f.params["age"] > 0 for f in fits.values()):
        models[species] = fits

len(models)

In [ ]:
def design(ages):
    ages = np.asarray(ages, dtype=float)
    return np.column_stack([np.ones(len(ages)), ages]) 

def predict_range(species, ages):
    X = design(ages)
    a = np.exp(models[species]["p10"].predict(X))
    b = np.exp(models[species]["p90"].predict(X))
    return np.minimum(a, b), np.maximum(a, b)

t = test[test["species"].isin(models)].copy()
for sp, g in t.groupby("species"):
    t.loc[g.index, "lo"], t.loc[g.index, "hi"] = predict_range(sp, g["age"])

t["inside"] = (t["area_m2"] >= t["lo"]) & (t["area_m2"] <= t["hi"])
t["inside"].mean()

In [ ]:
def pinball(y, yhat, q):
    d = np.asarray(y) - yhat
    return np.mean(np.maximum(q * d, (q - 1) * d))

rows = []
for sp, fits in models.items():
    tr, te = train[train["species"] == sp], test[test["species"] == sp]
    rows.append({
        "species": sp,
        "train_loss_p10": pinball(tr["log_area"], fits["p10"].predict(design(tr["age"])), 0.1),
        "test_loss_p10":  pinball(te["log_area"], fits["p10"].predict(design(te["age"])), 0.1),
        "train_loss_p90": pinball(tr["log_area"], fits["p90"].predict(design(tr["age"])), 0.9),
        "test_loss_p90":  pinball(te["log_area"], fits["p90"].predict(design(te["age"])), 0.9),
    })
loss = pd.DataFrame(rows)
loss.drop(columns="species").mean()

In [ ]:
given_size = {"S": 0, "M": 2, "L": 4} 

for fits in models.values():
    for f in fits.values():
        f.remove_data()

# save the models and metadata to a pickle file
with open("tree_canopy_growth_model.pkl", "wb") as f:
    pickle.dump({
        "models": models,  # model
        "valid_age_range": [3, 17], # valid age range for prediction
        "size_offset_years": given_size,  # size offset years for S, M, L
    }, f)

In [ ]:
# load the exported file back and predict the same way the backend will
with open("tree_canopy_growth_model.pkl", "rb") as f:
    MODEL = pickle.load(f)


def predict_canopy(species, years, size=None, start_width_m=None):
    fits = MODEL["models"].get(species.lower())

    # check if the input parameters are valid
    if fits is None:
        raise ValueError("unsupported species")
    if years < 0:
        raise ValueError("years must be >= 0")
    if (size is None) == (start_width_m is None):
        raise ValueError("give exactly one of size or start_width_m")

    if size is not None:
        if size not in MODEL["size_offset_years"]:
            raise ValueError("size must be S, M or L")
        age0 = float(MODEL["size_offset_years"][size])
    else:
        if start_width_m <= 0:
            raise ValueError("start_width_m must be > 0")
        a50, b50 = fits["p50"].params
        start_area = math.pi * (start_width_m / 2) ** 2
        age0 = max(0.0, (math.log(start_area) - a50) / b50)

    age = age0 + years
    X = np.array([[1.0, age]])
    lo = float(np.exp(fits["p10"].predict(X))[0])
    hi = float(np.exp(fits["p90"].predict(X))[0])
    lo, hi = min(lo, hi), max(lo, hi)

    min_age, max_age = MODEL["valid_age_range"]
    return {
        "canopy_m2_min": round(lo, 1),
        "canopy_m2_max": round(hi, 1),
        "equivalent_age_years": round(age, 1),
        "outside_training_range": not (min_age <= age <= max_age),
    }


species = "platanus x acerifolia"
pd.DataFrame([
    {"input": f"size={s}, years={y}", **predict_canopy(species, y, size=s)}
    for s in MODEL["size_offset_years"] for y in (3, 5, 10)
] + [
    {"input": "start_width_m=3.0, years=5", **predict_canopy(species, 5, start_width_m=3.0)}
])

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

QUANTILES = [("p10", 0.1), ("p50", 0.5), ("p90", 0.9)]

# model prediction vs baseline: same species, ignore age
for sp, g in t.groupby("species"):
    for name, q in QUANTILES:
        t.loc[g.index, name] = models[sp][name].predict(design(g["age"]))
        t.loc[g.index, name + "_base"] = train.loc[train["species"] == sp, "log_area"].quantile(q)

scores = {f"pseudo_r2_{name}": 1 - pinball(t["log_area"], t[name], q) / pinball(t["log_area"], t[name + "_base"], q)
          for name, q in QUANTILES}
scores["r2_p50_log"] = r2_score(t["log_area"], t["p50"])
scores["r2_p50_m2"] = r2_score(t["area_m2"], np.exp(t["p50"]))
scores["mae_p50_m2"] = mean_absolute_error(t["area_m2"], np.exp(t["p50"]))
pd.Series(scores).round(3)

## Port Phillip model

Same idea as above, but trained on the City of Port Phillip inventory, which records each tree's planting year and crown width directly (no canopy polygons). Crown is "width in metres of the tree's foliage" (OpenCouncilData trees standard); banded values like 4-5 m are stored as the midpoint.

Known limits of this data: planting years before 1991 are recorded to the decade (1980, 1990...), so ages above ~30 are +/- 5 years; the observation date is the inventory's last update, not necessarily a field measurement; about 3,300 trees have no update date and are dropped, most of them planted after 2010.

In [3]:
# use token to connect to the database
conn.close()
token = boto3.client("rds", region_name="ap-southeast-2").generate_db_auth_token(
    DBHostname=HOST, Port=5432, DBUsername=USER, Region="ap-southeast-2")
conn = psycopg.connect(host=HOST, port=5432, dbname="postgres", user=USER,
                       password=token, sslmode="require")


# get the latest dataset version ids for canopy and tree inventory
pp_version = query("""
    SELECT DISTINCT dataset_version_id FROM latest_metropolitan_named_tree_inventory
    WHERE municipality = 'City of Port Phillip'""").iloc[0, 0]

# query the City of Port Phillip tree inventory dataset
pp = query("""
    SELECT lower(scientific_name) AS species, year_planted, source_observed_on,
           canopy_width_m, height_m, diameter_breast_height_cm
    FROM named_tree_inventory
    WHERE dataset_version_id = %(version)s AND quality_status = 'passed'
    """, {"version": pp_version})
conn.close()

# length of the City of Port Phillip tree inventory dataset
len(pp)

45989

In [4]:
display(pp.head(10))

,species,year_planted,source_observed_on,canopy_width_m,height_m,diameter_breast_height_cm
0,lophostemon confertus,1995.0,2015-06-09,3.0,5.0,20.0
1,lophostemon confertus,1960.0,2015-06-22,8.0,10.0,65.0
2,quercus palustris,2012.0,2012-07-31,None,None,None
3,quercus palustris,2012.0,2012-07-31,None,None,None
4,quercus palustris,2012.0,2012-07-31,None,None,None
5,quercus palustris,2012.0,2012-07-31,None,None,None
6,punica granatum,2012.0,2012-07-31,None,None,None
7,citrus limon,2012.0,2012-07-31,None,None,None
8,prunus persica,2012.0,2012-07-31,None,None,None
9,citrus limon,2012.0,2012-07-31,None,None,None


In [5]:
# wrangling the dataset
# drop NA rows for species, year_planted, source_observed_on, and canopy_width_m
pp = pp.dropna(subset=["species", "year_planted", "source_observed_on", "canopy_width_m"]).copy()
pp["width_m"] = pp["canopy_width_m"].astype(float)
pp["age"] = pd.to_datetime(pp["source_observed_on"]).dt.year - pp["year_planted"]

# not a species: "to define" and genus-only names like "eucalyptus sp."
pp = pp[~pp["species"].str.contains(r" sp\.$|^to define$")]

# replace the row with 
placeholder = (pp["width_m"] == 1) & pp["height_m"].isna() & pp["diameter_breast_height_cm"].isna()
pp = pp[~placeholder]

pp = pp[(pp["year_planted"] >= 1950) & (pp["age"] >= 0)]   # drops the 1900 placeholder and older decade guesses
pp = pp[(pp["width_m"] > 0) & (pp["width_m"] <= 30)]
pp["area_m2"] = np.pi * (pp["width_m"] / 2) ** 2
pp["log_area"] = np.log(pp["area_m2"])

counts = pp["species"].value_counts()
pp = pp[pp["species"].isin(counts[counts >= 30].index)]

len(pp), pp["species"].nunique(), pp["age"].min(), pp["age"].max()

(32587, 134, np.float64(0.0), np.float64(70.0))

In [6]:
display(pp.head(10))

,species,year_planted,source_observed_on,canopy_width_m,height_m,diameter_breast_height_cm,width_m,age,area_m2,log_area
0,lophostemon confertus,1995.0,2015-06-09,3.0,5.0,20.0,3.0,20.0,7.068583,1.955660
1,lophostemon confertus,1960.0,2015-06-22,8.0,10.0,65.0,8.0,55.0,50.265482,3.917319
24,lophostemon confertus,1990.0,2015-06-09,4.0,5.0,30.0,4.0,25.0,12.566371,2.531024
25,platanus x acerifolia,1970.0,2015-06-12,8.0,7.0,35.0,8.0,45.0,50.265482,3.917319
26,platanus x acerifolia,1970.0,2015-06-12,9.0,9.0,40.0,9.0,45.0,63.617251,4.152885
27,platanus x acerifolia,1970.0,2015-06-12,7.5,9.0,40.0,7.5,45.0,44.178647,3.788242
28,platanus x acerifolia,1960.0,2015-06-12,10.0,10.0,45.0,10.0,55.0,78.539816,4.363606
29,platanus x acerifolia,1960.0,2017-07-10,8.5,10.0,40.0,8.5,57.0,56.745017,4.038568
30,platanus x acerifolia,1970.0,2015-06-12,7.0,10.0,35.0,7.0,45.0,38.484510,3.650256
31,melaleuca linariifolia,1980.0,2015-06-12,4.5,4.0,40.0,4.5,35.0,15.904313,2.766590


In [7]:
# canopy grows fast when young and slows later, so the feature is log(age + 1), not age
QUANTILES = [("p10", 0.1), ("p50", 0.5), ("p90", 0.9)]

# switch to log(age + 1) feature for the Port Phillip dataset
def pp_design(ages):
    ages = np.asarray(ages, dtype=float)
    return np.column_stack([np.ones(len(ages)), np.log1p(ages)])

# test train split for dataset 
pp_train, pp_test = train_test_split(pp, test_size=0.2, random_state=42, stratify=pp["species"])
age_grid = pp_design(np.arange(0, pp["age"].max() + 1))

pp_models = {}

# for each tree type, fit 10th, median and 90th quantile regressoin models
for species, group in pp_train.groupby("species"):
    X = pp_design(group["age"])
  
    with np.errstate(divide="ignore", invalid="ignore"):
        fits = {name: sm.QuantReg(group["log_area"].to_numpy(), X).fit(q=q, max_iter=5000)
                for name, q in QUANTILES}


    p10, p50, p90 = (fits[name].predict(age_grid) for name, _ in QUANTILES)
    if all(f.params[1] > 1e-3 for f in fits.values()) and (p10 <= p50).all() and (p50 <= p90).all():
        pp_models[species] = fits

# show the number of training and testing samples, the number of unique species in the training set, and the number of fitted models


print("Number of training samples:", len(pp_train))
print("Number of testing samples:", len(pp_test))
print("Number of unique species in training set:", pp_train["species"].nunique())
print("Number of fitted models:", len(pp_models))


Number of training samples: 26069
Number of testing samples: 6518
Number of unique species in training set: 134
Number of fitted models: 69


In [8]:
def pinball(y, yhat, q):
    d = np.asarray(y) - yhat
    return np.mean(np.maximum(q * d, (q - 1) * d))

def pp_predict(rows):
    rows = rows[rows["species"].isin(pp_models)].copy()
    for sp, g in rows.groupby("species"):
        for name, q in QUANTILES:
            rows.loc[g.index, name] = pp_models[sp][name].predict(pp_design(g["age"]))
            rows.loc[g.index, name + "_base"] = pp_train.loc[pp_train["species"] == sp, "log_area"].quantile(q)
    return rows

pt = pp_predict(pp_test)   # only species that have a model

# widths are banded, so many trees sit exactly on a line; the solver lands within ~1e-5 of them,
# so count trees within 1e-4 (0.01% of area) of a line as on it
pt["inside"] = (pt["log_area"] >= pt["p10"] - 1e-4) & (pt["log_area"] <= pt["p90"] + 1e-4)

pp_train_pred = pp_predict(pp_train)
pp_scores = {"scored_test_rows": len(pt),
             "coverage": pt["inside"].mean(),
             "coverage_age_0_14": pt.loc[pt["age"] <= 14, "inside"].mean(),
             "coverage_width_not_1m": pt.loc[pt["width_m"] != 1, "inside"].mean()}
for name, q in QUANTILES:
    pp_scores[f"train_loss_{name}"] = pinball(pp_train_pred["log_area"], pp_train_pred[name], q)
    pp_scores[f"test_loss_{name}"] = pinball(pt["log_area"], pt[name], q)
    pp_scores[f"pseudo_r2_{name}"] = 1 - pinball(pt["log_area"], pt[name], q) / pinball(pt["log_area"], pt[name + "_base"], q)
pp_scores["r2_p50_log"] = r2_score(pt["log_area"], pt["p50"])
pp_scores["r2_p50_m2"] = r2_score(pt["area_m2"], np.exp(pt["p50"]))
pp_scores["mae_p50_m2"] = mean_absolute_error(pt["area_m2"], np.exp(pt["p50"]))
pp_scores["median_area_m2"] = pt["area_m2"].median()
pd.Series(pp_scores).round(3)

scored_test_rows         4372.000
coverage                    0.807
coverage_age_0_14           0.787
coverage_width_not_1m       0.822
train_loss_p10              0.143
test_loss_p10               0.143
pseudo_r2_p10               0.406
train_loss_p50              0.285
test_loss_p50               0.286
pseudo_r2_p50               0.375
train_loss_p90              0.115
test_loss_p90               0.118
pseudo_r2_p90               0.273
r2_p50_log                  0.714
r2_p50_m2                   0.531
mae_p50_m2                 15.108
median_area_m2             19.635
dtype: float64

In [9]:
# London Plane for each S / M / L head start (the backend uses SIZE_OFFSET_YEARS in tree_growth.py)
size_offset = {"S": 0, "M": 2, "L": 10}
rows = []
for size, offset in size_offset.items():
    for years in (3, 5, 10):
        X = pp_design([offset + years])
        p10, p50, p90 = (float(np.exp(pp_models["platanus x acerifolia"][name].predict(X))[0])
                         for name, _ in QUANTILES)
        rows.append({"size": size, "years": years, "canopy_m2_min": round(p10, 1),
                     "canopy_m2_median": round(p50, 1), "canopy_m2_max": round(p90, 1)})
pd.DataFrame(rows)

,size,years,canopy_m2_min,canopy_m2_median,canopy_m2_max
0,S,3,0.6,2.0,6.6
1,S,5,1.0,3.6,10.6
2,S,10,2.7,8.2,21.4
3,M,3,1.0,3.6,10.6
4,M,5,1.6,5.3,14.8
5,M,10,3.4,10.3,26.0
6,L,3,3.8,11.4,28.4
7,L,5,4.7,13.7,33.2
8,L,10,7.2,19.9,45.6


In [10]:
# ---------- height model ----------
# same method as the crown model above; the only difference is that we predict tree height, not canopy area

# 1. keep trees that have a height, and drop typos above 50 m (one tree is recorded as 2000 m)
height_train = pp_train.copy()
height_train["height"] = height_train["height_m"].astype(float)
height_train = height_train[(height_train["height"] > 0) & (height_train["height"] <= 50)]
height_train["log_height"] = np.log(height_train["height"])

height_test = pp_test.copy()
height_test["height"] = height_test["height_m"].astype(float)
height_test = height_test[(height_test["height"] > 0) & (height_test["height"] <= 50)]
height_test["log_height"] = np.log(height_test["height"])

# 2. for each species, fit three lines: ln(height) = a + b * ln(age + 1) at p10, p50 and p90
height_models = {}
for species, group in height_train.groupby("species"):
    if len(group) < 30:   # too few trees with a height for this species
        continue
    X = pp_design(group["age"])
    with np.errstate(divide="ignore", invalid="ignore"):
        fits = {name: sm.QuantReg(group["log_height"].to_numpy(), X).fit(q=q, max_iter=5000)
                for name, q in QUANTILES}

    # 3. keep the species only if all three lines grow and never cross (same checks as the crown)
    p10, p50, p90 = (fits[name].predict(age_grid) for name, _ in QUANTILES)
    if all(f.params[1] > 1e-3 for f in fits.values()) and (p10 <= p50).all() and (p50 <= p90).all():
        height_models[species] = fits

print("Number of height models:", len(height_models))

C:\Users\13282\AppData\Local\Temp\ipykernel_1608\1538150641.py:22: IterationLimitWarning: Maximum number of iterations (5000) reached.
  fits = {name: sm.QuantReg(group["log_height"].to_numpy(), X).fit(q=q, max_iter=5000)


Number of height models: 74


In [11]:
# ---------- DBH model (trunk diameter at breast height, in cm) ----------
# same steps as the height model

# 1. keep trees that have a DBH, and drop typos above 500 cm (two trees have a year such as 2008 typed in)
dbh_train = pp_train.copy()
dbh_train["dbh"] = dbh_train["diameter_breast_height_cm"].astype(float)
dbh_train = dbh_train[(dbh_train["dbh"] > 0) & (dbh_train["dbh"] <= 500)]
dbh_train["log_dbh"] = np.log(dbh_train["dbh"])

dbh_test = pp_test.copy()
dbh_test["dbh"] = dbh_test["diameter_breast_height_cm"].astype(float)
dbh_test = dbh_test[(dbh_test["dbh"] > 0) & (dbh_test["dbh"] <= 500)]
dbh_test["log_dbh"] = np.log(dbh_test["dbh"])

# 2. for each species, fit three lines: ln(DBH) = a + b * ln(age + 1) at p10, p50 and p90
dbh_models = {}
for species, group in dbh_train.groupby("species"):
    if len(group) < 30:   # too few trees with a DBH for this species
        continue
    X = pp_design(group["age"])
    with np.errstate(divide="ignore", invalid="ignore"):
        fits = {name: sm.QuantReg(group["log_dbh"].to_numpy(), X).fit(q=q, max_iter=5000)
                for name, q in QUANTILES}

    # 3. keep the species only if all three lines grow and never cross
    p10, p50, p90 = (fits[name].predict(age_grid) for name, _ in QUANTILES)
    if all(f.params[1] > 1e-3 for f in fits.values()) and (p10 <= p50).all() and (p50 <= p90).all():
        dbh_models[species] = fits

print("Number of DBH models:", len(dbh_models))

C:\Users\13282\AppData\Local\Temp\ipykernel_1608\1638928636.py:22: IterationLimitWarning: Maximum number of iterations (5000) reached.
  fits = {name: sm.QuantReg(group["log_dbh"].to_numpy(), X).fit(q=q, max_iter=5000)


Number of DBH models: 86


In [15]:
# score the height and DBH models on the test trees (same scores as the crown model)
def score(models, train, test, log_column):
    test = test[test["species"].isin(models)].copy()   # only species that have a model
    for sp, g in test.groupby("species"):
        for name, q in QUANTILES:
            test.loc[g.index, name] = models[sp][name].predict(pp_design(g["age"]))
            # baseline for pseudo-R2: the same species' quantile, ignoring age
            test.loc[g.index, name + "_base"] = train.loc[train["species"] == sp, log_column].quantile(q)

    y = test[log_column]
    inside = (y >= test["p10"] - 1e-4) & (y <= test["p90"] + 1e-4)
    return {"species": len(models),
            "test_rows": len(test),
            "coverage": inside.mean(),
            "pseudo_r2_p50": 1 - pinball(y, test["p50"], 0.5) / pinball(y, test["p50_base"], 0.5),
            "r2_p50_log": r2_score(y, test["p50"]),
            "mae_p50": mean_absolute_error(np.exp(y), np.exp(test["p50"])),
            "median": np.exp(y).median()}

pd.DataFrame({"height_m": score(height_models, height_train, height_test, "log_height"),
              "dbh_cm": score(dbh_models, dbh_train, dbh_test, "log_dbh")}).round(3)

,height_m,dbh_cm
species,74.000,86.000
test_rows,5233.000,5357.000
coverage,0.816,0.810
pseudo_r2_p50,0.281,0.417
r2_p50_log,0.671,0.785
mae_p50,1.341,8.188
median,6.000,25.000


In [16]:
# export the Port Phillip models straight into the backend
# every line is saved as two plain numbers [a, b], so the backend can predict with the math module alone

def to_numbers(fits):
    # {"p10": [a, b], "p50": [a, b], "p90": [a, b]}
    return {name: [float(v) for v in fits[name].params] for name, _ in QUANTILES}

export = {"models": {}}
for species, fits in pp_models.items():
    tree = {}
    tree["max_age"] = int(pp_train.loc[pp_train["species"] == species, "age"].max())   # oldest training tree
    tree["canopy_m2"] = to_numbers(fits)
    if species in height_models:   # left out when this species has no reliable height model
        tree["height_m"] = to_numbers(height_models[species])
    if species in dbh_models:      # left out when this species has no reliable DBH model
        tree["dbh_cm"] = to_numbers(dbh_models[species])
    export["models"][species] = tree

with open("../backend/app/greening_model/config/tree_canopy_growth_model.pkl", "wb") as f:
    pickle.dump(export, f)


print("Exported to ../backend/app/greening_model/config/tree_canopy_growth_model.pkl")
print("Number of species models exported:", len(export["models"]))


Exported to ../backend/app/greening_model/config/tree_canopy_growth_model.pkl
Number of species models exported: 69
